In [50]:
import pandas as pd           # outil pour manipuler les tableaux de donnees (comme Excel mais en code)
from pathlib import Path      # outil pour manipuler les chemins de fichiers
from datetime import datetime # outil pour travailler avec les dates et heures


In [51]:
# Nom du fichier CSV qu'on va lire en entree (genere par le script de scan)
FICHIER_ENTREE  = "raw_file_metadata.csv"

# Nom du fichier CSV qu'on va produire a la fin du nettoyage
FICHIER_SORTIE  = "cleaned_file_metadata.csv"

# On note la date et l'heure actuelles
# Elle sert de reference pour calculer l'age des fichiers et les jours depuis la derniere modification
DATE_REFERENCE = datetime.now()

# Liste des extensions qu'on considere inutiles et qu'on va supprimer du tableau
EXTENSIONS_A_SUPPRIMER = {".tmp", ".log", ".ini", ".sys", ".lnk"}

# Dictionnaire qui associe chaque categorie de fichier a la liste de ses extensions
CARTE_CATEGORIES = {
    "document" : {".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx", ".txt", ".odt"},
    "image"    : {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".svg", ".webp", ".tiff", ".ico"},
    "video"    : {".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv", ".webm"},
    "audio"    : {".mp3", ".wav", ".aac", ".flac", ".ogg", ".wma"},
    "code"     : {".py", ".js", ".ts", ".java", ".c", ".cpp", ".cs", ".html", ".css", ".sql"},
    "archive"  : {".zip", ".rar", ".7z", ".tar", ".gz", ".bz2"},
    "donnees"  : {".csv", ".json", ".xml", ".yaml", ".yml", ".db", ".sqlite"},
}

In [52]:
def obtenir_categorie(extension):
    # On parcourt chaque categorie et la liste d'extensions associee
    for categorie, extensions in CARTE_CATEGORIES.items():
        # Si l'extension du fichier fait partie de cette liste
        if extension in extensions:
            return categorie
    # Si aucune categorie ne correspond, on retourne "autre"
    return "autre"

In [53]:
def obtenir_repertoire(chemin):
    try:
        # On decompose le chemin en morceaux (un morceau par dossier)
        parties = Path(str(chemin)).parts
        # On retourne le deuxieme morceau (le premier dossier apres la racine)
        return parties[1] if len(parties) > 1 else "racine"
    except Exception:
        return "inconnu"

In [54]:
def obtenir_dossier_parent(chemin):
    try:
        # .parent donne le dossier qui contient le fichier
        return str(Path(chemin).parent)
    except Exception:
        return "inconnu"

In [55]:
def calculer_profondeur(chemin):
    try:
        # .parts donne la liste des dossiers du chemin du parent
        # On soustrait 1 pour ne pas compter la racine
        return len(Path(chemin).parent.parts) - 1
    except Exception:
        return 0

In [56]:
def categoriser_taille(taille_octets):
    # Moins de 1 Mo -> Small
    if taille_octets < 1_048_576:
        return "Small"
    # Entre 1 Mo et 100 Mo -> Medium
    elif taille_octets < 104_857_600:
        return "Medium"
    # Plus de 100 Mo -> Large
    else:
        return "Large"

In [57]:
def formater_taille_lisible(taille_octets):
    # Si la taille depasse 1 Go, on affiche en GB
    if taille_octets >= 1_073_741_824:
        return f"{taille_octets / 1_073_741_824:.2f} GB"
    # Si la taille depasse 1 Mo, on affiche en MB
    elif taille_octets >= 1_048_576:
        return f"{taille_octets / 1_048_576:.2f} MB"
    # Sinon on affiche en KB
    else:
        return f"{taille_octets / 1_024:.2f} KB"

In [58]:
def calculer_age_fichier(date_creation):
    try:
        # On soustrait la date de creation a la date de reference (aujourd'hui)
        delta = DATE_REFERENCE - date_creation.replace(tzinfo=None)
        # On retourne le nombre de jours, jamais negatif
        return max(delta.days, 0)
    except Exception:
        return -1

In [59]:
def calculer_jours_depuis_modif(date_modification):
    try:
        delta = DATE_REFERENCE - date_modification.replace(tzinfo=None)
        return max(delta.days, 0)
    except Exception:
        return -1 

In [60]:
df = pd.read_csv(FICHIER_ENTREE, dtype=str)
df.head()

,identifiant,nom_fichier,extension,chemin,taille_octets,date_creation,date_modification,hash_md5,owner
0,1,DumpStack.log.tmp,.tmp,C:\DumpStack.log.tmp,12288,2026-06-01T08:02:22.418720,2026-06-24T06:57:03.545696,NaN,AOmpa
1,2,hiberfil.sys,.sys,C:\hiberfil.sys,6739701760,2026-06-01T08:05:03.438314,2026-06-24T06:56:56.679811,NaN,AOmpa
2,3,LAPSCheck.log,.log,C:\LAPSCheck.log,5165,2026-06-01T09:09:44.965019,2026-06-24T05:15:07.886977,40b9dc89458fffe007aecb429bdbb9af,AOmpa
3,4,pagefile.sys,.sys,C:\pagefile.sys,36507222016,2026-06-01T08:02:22.266039,2026-06-24T06:57:03.529561,NaN,AOmpa
4,5,swapfile.sys,.sys,C:\swapfile.sys,16777216,2026-06-01T08:02:22.418720,2026-06-24T06:57:03.545696,NaN,AOmpa


In [61]:
def nettoyer_donnees(chemin_entree, chemin_sortie):

    # On charge le CSV brut dans un tableau (DataFrame)
    df = pd.read_csv(chemin_entree, dtype=str)

    # On garde le nombre de lignes de depart pour le bilan final
    total = len(df)
    print(f"Lignes chargees : {total}")

    # On supprime les lignes ou une information essentielle est manquante
    df = df.dropna(subset=["nom_fichier", "chemin", "hash_md5", "taille_octets"])

    df = df.copy()

    # On convertit la colonne taille_octets en nombre
    df["taille_octets"] = pd.to_numeric(df["taille_octets"], errors="coerce")

    # On garde uniquement les fichiers dont la taille est superieure a 0
    df = df[df["taille_octets"] > 0]

    # On garde uniquement les lignes dont le hash MD5 a la bonne longueur (32 caracteres)
    df = df[df["hash_md5"].str.strip().str.len() == 32]

    # On convertit les colonnes de dates en vrais objets date
    df["date_creation"]     = pd.to_datetime(df["date_creation"],     errors="coerce")
    df["date_modification"] = pd.to_datetime(df["date_modification"], errors="coerce")

    # On garde uniquement les lignes coherentes (modif apres creation)
    df = df[df["date_modification"] >= df["date_creation"]]

    # On nettoie la colonne extension : espaces enleves, mise en minuscule
    df["extension"] = df["extension"].str.strip().str.lower()

    # On supprime les lignes dont l'extension fait partie de la liste a ignorer
    df = df[~df["extension"].isin(EXTENSIONS_A_SUPPRIMER)]

    # Supression des lignes ou les fichier sont sans extension
    df=df.dropna(subset=["extension"])

    # On enleve les espaces inutiles autour du nom de fichier
    df["nom_fichier"] = df["nom_fichier"].str.strip()

    # On supprime les doublons de chemin
    df = df.drop_duplicates(subset=["chemin"])

    # On ajoute la taille en kilo-octets, arrondie a 2 decimales
    df["taille_ko"]  = (df["taille_octets"] / 1024).round(2)

    # On ajoute la categorie du fichier
    df["categorie"]  = df["extension"].apply(obtenir_categorie)

    # On ajoute le repertoire racine du fichier
    df["repertoire"] = df["chemin"].apply(obtenir_repertoire)

    # On ajoute le chemin du dossier parent
    df["folder_path"]             = df["chemin"].apply(obtenir_dossier_parent)

    # On ajoute la profondeur du fichier dans l'arborescence
    df["folder_depth"]            = df["chemin"].apply(calculer_profondeur)

    # On ajoute la categorie de taille (Small, Medium, Large)
    df["size_category"]           = df["taille_octets"].apply(categoriser_taille)

    # On ajoute l'age du fichier en jours
    df["file_age"]                = df["date_creation"].apply(calculer_age_fichier)

    # On ajoute le nombre de jours depuis la derniere modification
    df["days_since_modification"] = df["date_modification"].apply(calculer_jours_depuis_modif)

    # On ajoute la taille du fichier en texte lisible
    df["file_size_readable"]      = df["taille_octets"].apply(formater_taille_lisible)

    # On reconvertit les dates en texte lisible
    df["date_creation"]     = pd.to_datetime(df["date_creation"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
    df["date_modification"] = pd.to_datetime(df["date_modification"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

    # On definit l'ordre final des colonnes
    colonnes_ordonnees = [
        "identifiant", "nom_fichier", "extension", "chemin",
        "taille_octets", "date_creation", "date_modification",
        "hash_md5", "owner",
        "taille_ko", "categorie", "repertoire",
        "folder_path", "folder_depth", "size_category",
        "file_age", "days_since_modification", "file_size_readable",
    ]

    # On reorganise le tableau selon cet ordre
    df = df[colonnes_ordonnees]

    # On exporte le tableau nettoye dans un nouveau fichier CSV
    df.to_csv(chemin_sortie, index=False, encoding="utf-8")

    # On affiche le bilan final
    print(f"Fichier exporte    : {chemin_sortie}")
    print(f"Lignes totales     : {total}")
    print(f"Lignes conservees  : {len(df)}")
    print(f"Lignes supprimees  : {total - len(df)}")
    print(f"Colonnes totales   : {len(df.columns)}")

    return df 

In [62]:
df_clean = nettoyer_donnees(FICHIER_ENTREE, FICHIER_SORTIE)
df_clean.head()

Lignes chargees : 297492
Fichier exporte    : cleaned_file_metadata.csv
Lignes totales     : 297492
Lignes conservees  : 197323
Lignes supprimees  : 100169
Colonnes totales   : 18


,identifiant,nom_fichier,extension,chemin,taille_octets,date_creation,date_modification,hash_md5,owner,taille_ko,categorie,repertoire,folder_path,folder_depth,size_category,file_age,days_since_modification,file_size_readable
5,6,$I3VK3SL.txt,.txt,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,172,2026-06-19 02:35:08,2026-06-19 02:35:08,08a5cfaeebc5a6c2de35b3e7b4a27da2,AOmpa,0.17,document,$Recycle.Bin,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,2,Small,5,5,0.17 KB
6,7,$I4IO8D5.py,.py,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,168,2026-06-11 07:30:53,2026-06-11 07:30:53,34cf3317db8e34f2301bcfc84636cf76,AOmpa,0.16,code,$Recycle.Bin,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,2,Small,13,13,0.16 KB
7,8,$I92K7IK.txt,.txt,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,172,2026-06-19 02:35:04,2026-06-19 02:35:04,d9dfb2eeb3d15671f1d60a9d31cabe69,AOmpa,0.17,document,$Recycle.Bin,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,2,Small,5,5,0.17 KB
8,9,$I9XHEMT.txt,.txt,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,172,2026-06-19 02:35:11,2026-06-19 02:35:11,879ec5bc60780ae15ce465989fdae022,AOmpa,0.17,document,$Recycle.Bin,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,2,Small,5,5,0.17 KB
9,10,$ICOVVF5.csv,.csv,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,178,2026-06-17 03:13:32,2026-06-17 03:13:32,b69ec74942290037f98c60de63597510,AOmpa,0.17,donnees,$Recycle.Bin,C:\$Recycle.Bin\S-1-12-1-2125803670-1098790677...,2,Small,7,7,0.17 KB


In [63]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 197323 entries, 5 to 297490
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   identifiant              197323 non-null  str    
 1   nom_fichier              197323 non-null  str    
 2   extension                197323 non-null  str    
 3   chemin                   197323 non-null  str    
 4   taille_octets            197323 non-null  int64  
 5   date_creation            197323 non-null  str    
 6   date_modification        197323 non-null  str    
 7   hash_md5                 197323 non-null  str    
 8   owner                    197323 non-null  str    
 9   taille_ko                197323 non-null  float64
 10  categorie                197323 non-null  str    
 11  repertoire               197323 non-null  str    
 12  folder_path              197323 non-null  str    
 13  folder_depth             197323 non-null  int64  
 14  size_category       